# 課堂實作：你的第一支量子程式

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_GITHUB_ID/quantum-intro-course/blob/main/01_hello_quantum.ipynb)

今天我們要親手做出一個**糾纏**的量子電路，並在模擬器上執行，看看疊加與糾纏在結果中長什麼樣子。

> ⚠️ **開始前請先做這件事**：點選上方選單 **「檔案 → 在雲端硬碟中儲存副本」**。
> 從 GitHub 開啟的筆記本不會保存你的修改，存一份副本到自己的雲端硬碟，回家作業才接得下去。

**怎麼執行程式？** 點一下程式碼格子，按 `Shift + Enter`（或格子左邊的 ▶️），就會執行並跳到下一格。請**由上往下依序執行**。

| 步驟 | 內容 | 時間 |
|---|---|---|
| 0 | 安裝與環境檢查 | 5 分 |
| 1 | 用 Composer 看見電路（講師示範） | 7 分 |
| 2 | 用程式寫出 Bell 態，在模擬器上執行 | 7 分 |
| 3 | 小實驗：shots 次數的影響／雜訊模擬（擇一） | 5 分 |
| 4 | 挑戰題：做出其他 Bell 態 | 6 分 |

---
## 步驟 0：安裝與環境檢查

In [ ]:
# 安裝 Qiskit（量子程式套件）、Aer（模擬器）、IBM Runtime（連線 IBM 量子電腦用）
# 約需 1～2 分鐘，看到最後沒有紅色錯誤訊息就代表成功
!pip install -q 'qiskit[visualization]' qiskit-aer qiskit-ibm-runtime

In [ ]:
import sys, warnings
warnings.filterwarnings("ignore")   # 隱藏套件更新提示，不影響執行結果

import qiskit, qiskit_aer, qiskit_ibm_runtime

from qiskit import QuantumCircuit
from qiskit.visualization import plot_histogram
from qiskit_aer import AerSimulator
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2 as Sampler

print("Python            :", sys.version.split()[0])
print("qiskit            :", qiskit.__version__)
print("qiskit-aer        :", qiskit_aer.__version__)
print("qiskit-ibm-runtime:", qiskit_ibm_runtime.__version__)
print("環境準備完成！")
print("（圖表的文字使用英文，是因為 Colab 預設字型不支援中文）")

> 💡 如果這一格出現錯誤，通常是上一格的安裝沒有完成。請重新執行安裝那一格，再執行這一格。
> 若仍有問題，可以點選「執行階段 → 重新啟動工作階段」後，從頭再執行一次。

---
## 步驟 1：用 Composer 看見電路

[IBM Quantum Composer](https://quantum.cloud.ibm.com/composer) 可以用拖拉的方式組出量子電路，不用寫程式。

示範中請留意三件事：

- **電路圖怎麼看**：每一條橫線代表一個量子位元，線上的方塊或符號代表作用在它身上的「閘」。
- **H 閘（Hadamard）**：讓量子位元進入**疊加態**，測量時 0 和 1 都有可能出現。
- **CNOT 閘**：連接兩個量子位元，讓它們**糾纏**，兩者的測量結果會緊密關聯。

---
## 步驟 2：用程式寫出 Bell 態

剛才在 Composer 拖出來的電路，現在用 Python 寫一次。
這個電路做出的狀態叫做 **Φ⁺（phi-plus）Bell 態**，是最經典的兩量子位元糾纏態。

為什麼要學寫程式？因為量子位元一多、電路一複雜，就不可能再用拖拉的方式完成了。

In [ ]:
bell = QuantumCircuit(2)   # 準備 2 個量子位元，一開始都是 0
bell.h(0)                  # H 閘：讓量子位元 0 進入疊加態
bell.cx(0, 1)              # CNOT 閘：以 0 為控制、1 為目標，讓兩者糾纏
bell.measure_all()         # 測量全部量子位元

bell.draw("mpl")

接著定義一個**小幫手函式**，負責「送出電路 → 執行 → 取回結果」這一連串步驟。
這一格只是定義，執行後不會有輸出，這是正常的。

In [ ]:
def run_circuit_and_get_counts(circuit, backend, shots=1000):
    """把電路送到指定的後端（模擬器或真機）執行，回傳每種結果出現的次數。"""
    # 1. 轉譯：把電路改寫成這個後端看得懂、能有效率執行的形式
    pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
    isa_circuit = pm.run(circuit)
    # 2. 用 Sampler 執行指定的次數（shots）
    sampler = Sampler(mode=backend)
    job = sampler.run([isa_circuit], shots=shots)
    # 3. 取回結果並整理成 {'00': 次數, '11': 次數, ...}
    return job.result()[0].data.meas.get_counts()

### 在模擬器上執行

模擬器是用一般電腦模擬「完美、沒有雜訊」的量子電腦。我們讓電路跑 1000 次（1000 shots），統計每種結果出現幾次。

In [ ]:
simulator = AerSimulator()
counts = run_circuit_and_get_counts(bell, simulator, shots=1000)

print(counts)
plot_histogram(counts)

### 🤔 想一想

<details>
<summary>1. 直方圖中只出現哪兩種結果？這代表什麼？</summary>

只有 `00` 和 `11`，各約一半。也就是說：一個量子位元測到 0，另一個一定也是 0；測到 1，另一個也一定是 1。
兩者的結果完全同步，這就是**糾纏**的特徵。
</details>

<details>
<summary>2. 既然模擬器是「完美」的，為什麼 00 和 11 的次數不會剛好各 500 次？</summary>

因為測量結果本來就是隨機的。就像擲一枚公正的硬幣 1000 次，也不會剛好 500 次正面、500 次反面。
模擬器完美模擬了這個隨機過程，所以每次執行的次數都會有一點點差異。
</details>

<details>
<summary>3. 只測量一次，看得到「疊加」嗎？</summary>

看不到。每次測量只會得到一個確定的 0 或 1。疊加只會在「重複很多次之後的統計」中顯現出來，這就是為什麼需要很多 shots。
</details>

---
## 步驟 3：小實驗

### 3A：shots 次數的影響

把同一個電路分別跑 10、100、1000 次，比較結果。

In [ ]:
results = []
for shots in [10, 100, 1000]:
    c = run_circuit_and_get_counts(bell, simulator, shots=shots)
    # 換算成比例，三種次數才放得進同一張圖比較
    results.append({k: v / shots for k, v in c.items()})
    print(f"{shots:>5} shots →", c)

plot_histogram(results, legend=["10 shots", "100 shots", "1000 shots"],
               title="More shots -> closer to 50 : 50")

👉 **觀察**：shots 越少，`00` 和 `11` 的比例可能差很多；shots 越多，越接近各一半。
量子電腦的答案是「統計出來的」，所以 shots 數要在**準確度**和**時間／成本**之間取捨。

### 3B：預先體驗真機的雜訊

真實的量子電腦會受到雜訊干擾。Qiskit 內建了「假後端」，可以模擬某台 IBM 真機的雜訊特性，不需要帳號也不用排隊。

In [ ]:
from qiskit_ibm_runtime.fake_provider import FakeManilaV2

noisy_simulator = AerSimulator.from_backend(FakeManilaV2())   # 模擬 IBM 舊機台 Manila 的雜訊
noisy_counts = run_circuit_and_get_counts(bell, noisy_simulator, shots=1000)
ideal_counts = run_circuit_and_get_counts(bell, simulator, shots=1000)

print("理想模擬器:", ideal_counts)
print("含雜訊模擬:", noisy_counts)
plot_histogram([ideal_counts, noisy_counts], legend=["Ideal simulator", "Noisy simulator"])

👉 **觀察**：含雜訊的版本中出現了少量的 `01` 和 `10`。在完美的世界裡它們不該出現，這些就是閘操作或測量出錯造成的。

📝 **回家驗證**：真正的 IBM 量子電腦跑出來的結果，會比較像哪一個？這就是今天的回家作業！

---
## 步驟 4：挑戰題 🏆 做出其他 Bell 態

Bell 態一共有四種。你已經做出 Φ⁺ 了，試著修改電路，做出另外三種：

| 名稱 | 測量時主要出現的結果 | 提示 |
|---|---|---|
| Φ⁺ | `00`、`11` | 你已經做出來了！ |
| Φ⁻ | `00`、`11` | 和 Φ⁺ 只差一個「正負號」，在 H 之後加一個 `z` 閘 |
| Ψ⁺ | `01`、`10` | 一開始先用 `x` 閘把其中一個量子位元翻成 1 |
| Ψ⁻ | `01`、`10` | 結合上面兩個提示（下一堂課會用到它！） |

可用的閘：`qc.x(i)`（翻轉 0↔1）、`qc.z(i)`（改變正負號）、`qc.h(i)`、`qc.cx(控制, 目標)`

In [ ]:
# 在這裡寫出你的 Ψ⁻ 電路
psi_minus = QuantumCircuit(2)
# psi_minus.???
# psi_minus.???
psi_minus.h(0)
psi_minus.cx(0, 1)
psi_minus.measure_all()

display(psi_minus.draw("mpl"))
plot_histogram(run_circuit_and_get_counts(psi_minus, simulator))

<details>
<summary>💡 偷看參考答案</summary>

```python
# Φ⁻：H 之後加 Z
phi_minus = QuantumCircuit(2)
phi_minus.h(0); phi_minus.z(0); phi_minus.cx(0, 1)

# Ψ⁺：先把量子位元 1 翻成 1
psi_plus = QuantumCircuit(2)
psi_plus.x(1); psi_plus.h(0); psi_plus.cx(0, 1)

# Ψ⁻：兩個量子位元都先翻成 1
psi_minus = QuantumCircuit(2)
psi_minus.x(0); psi_minus.x(1); psi_minus.h(0); psi_minus.cx(0, 1)
```

🤯 **想一想**：Φ⁺ 和 Φ⁻ 的直方圖幾乎一模一樣，Ψ⁺ 和 Ψ⁻ 也是。
「正負號」是真實存在的量子性質，卻無法從這種測量直接看出來，要用其他測量方式才分辨得出。
</details>

---
## 🎉 恭喜完成！

今天你做到了：

- ✅ 用 H 閘做出**疊加**、用 CNOT 閘做出**糾纏**
- ✅ 理解為什麼量子程式要跑很多 **shots**
- ✅ 看到理想模擬器和含**雜訊**結果的差別

**下一步** → 回家作業：[在真實的 IBM 量子電腦上執行](https://colab.research.google.com/github/YOUR_GITHUB_ID/quantum-intro-course/blob/main/02_homework_real_qpu.ipynb)

<sub>本實作改寫自 IBM Quantum Learning 課程〈Build and run your first quantum program〉：
https://quantum.cloud.ibm.com/learning/en/courses/use-a-qc-today/build-and-run-your-first-quantum-program</sub>